In [12]:
# ============================================================
# CP1 / S5 - PIPELINE DE INGESTIE MIT-BIH
# ============================================================

!pip -q install wfdb

from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import wfdb
import numpy as np
from collections import Counter

# ============================================================
# 1. INCARCAREA SI VERIFICAREA BAZEI DE DATE
# ============================================================

ZIP_PATH = (
    "/content/drive/MyDrive/"
    "mit-bih-arrhythmia-database-1.0.0.zip"
)
EXTRACT_ROOT = "/content/mit_bih_extracted"

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Arhiva nu exista: {ZIP_PATH}")


def find_data_dir(search_root):
    for root, dirs, files in os.walk(search_root):
        required = [
            f"{rec}.{ext}"
            for rec in ["100", "101"]
            for ext in ["hea", "dat", "atr"]
        ]
        if all(f in files for f in required):
            return root
    return None


os.makedirs(EXTRACT_ROOT, exist_ok=True)
DATA_DIR = find_data_dir(EXTRACT_ROOT)

if DATA_DIR is None:
    print("Se verifica si se extrage arhiva...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            raise RuntimeError(f"Arhiva este corupta: {bad_file}")
        zf.extractall(EXTRACT_ROOT)

    DATA_DIR = find_data_dir(EXTRACT_ROOT)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Nu s-a gasit directorul cu fisierele MIT-BIH."
    )

print(f"Director date: {DATA_DIR}")

# ============================================================
# 2. SEPARAREA IN TRAIN SI TEST
# 201 si 202 sunt pastrate impreuna in TRAIN
# ============================================================

DS1_TRAIN = [
    '101', '106', '108', '109', '112', '114', '115', '116',
    '118', '119', '122', '124', '201', '202', '203', '205',
    '207', '208', '209', '215', '220', '223', '230'
]

DS2_TEST = [
    '100', '103', '105', '111', '113', '117', '121', '123',
    '200', '210', '212', '213', '214', '219', '221', '222',
    '228', '231', '232', '233', '234'
]

assert set(DS1_TRAIN).isdisjoint(set(DS2_TEST))
assert {'201', '202'}.issubset(set(DS1_TRAIN))
assert len(DS1_TRAIN) == 23
assert len(DS2_TEST) == 21
assert len(set(DS1_TRAIN + DS2_TEST)) == 44

ALL_RECORDS = DS1_TRAIN + DS2_TEST

# Verificam prezenta fisierelor fiecarei inregistrari
for rec_id in ALL_RECORDS:
    for ext in ['hea', 'dat', 'atr']:
        path = os.path.join(DATA_DIR, f"{rec_id}.{ext}")
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Fisier lipsa: {path}")

print("Verificarea fisierelor a reusit.")
print(f"Train: {len(DS1_TRAIN)} inregistrari")
print(f"Test:  {len(DS2_TEST)} inregistrari")

# ============================================================
# 3. MAPAREA ETICHETELOR IN CLASELE AAMI
# ============================================================

AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q'
}

# ============================================================
# 4. EXTRAGEREA BATAILOR ECG SI A INTERVALELOR RR
# ============================================================

def extract_beats_from_record(
    record_path,
    win_before=90,
    win_after=110
):
    record = wfdb.rdrecord(record_path)
    ann = wfdb.rdann(record_path, 'atr')

    # Selectam canalul MLII dupa nume
    lead_names = [
        name.strip().upper()
        for name in record.sig_name
    ]

    if "MLII" not in lead_names:
        raise ValueError(
            f"Canalul MLII lipseste din {record_path}. "
            f"Canale disponibile: {record.sig_name}"
        )

    lead_idx = lead_names.index("MLII")
    signal = record.p_signal[:, lead_idx]
    fs = record.fs

    if not np.isclose(fs, 360.0):
        raise ValueError(
            f"Frecventa neasteptata pentru {record_path}: {fs} Hz"
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Semnalul contine valori NaN/Inf: {record_path}"
        )

    beats, labels = [], []
    rr_pre, rr_post = [], []

    # Pastram adnotarile care pot fi mapate in clasele AAMI
    valid_indices = [
        i for i, sym in enumerate(ann.symbol)
        if sym in AAMI_MAP
    ]

    for pos in range(1, len(valid_indices) - 1):
        i = valid_indices[pos]
        r_peak = ann.sample[i]

        start = r_peak - win_before
        end = r_peak + win_after

        # Fereastra trebuie sa fie completa
        if start < 0 or end > len(signal):
            continue

        beat = signal[start:end]

        if len(beat) != 200:
            continue

        beats.append(beat)
        labels.append(AAMI_MAP[ann.symbol[i]])

        # Intervalele RR sunt exprimate in secunde
        rr_pre.append(
            (r_peak - ann.sample[valid_indices[pos - 1]]) / fs
        )
        rr_post.append(
            (ann.sample[valid_indices[pos + 1]] - r_peak) / fs
        )

    if not beats:
        raise ValueError(
            f"Nu s-au extras batai din {record_path}"
        )

    return (
        np.asarray(beats, dtype=np.float32),
        np.asarray(labels),
        np.asarray(rr_pre, dtype=np.float32),
        np.asarray(rr_post, dtype=np.float32)
    )

# ============================================================
# 5. INCARCAREA UNUI SET DE DATE
# ============================================================

def load_split(data_dir, record_list, split_name):
    all_beats, all_labels = [], []
    all_rr_pre, all_rr_post = [], []

    for idx, rec_id in enumerate(record_list, start=1):
        print(
            f"{split_name}: {idx}/{len(record_list)} - {rec_id}",
            flush=True
        )

        record_path = os.path.join(data_dir, rec_id)

        beats, labels, rr_pre, rr_post = (
            extract_beats_from_record(record_path)
        )

        all_beats.append(beats)
        all_labels.append(labels)
        all_rr_pre.append(rr_pre)
        all_rr_post.append(rr_post)

    return (
        np.vstack(all_beats),
        np.concatenate(all_labels),
        np.concatenate(all_rr_pre),
        np.concatenate(all_rr_post)
    )

# ============================================================
# 6. EXECUTAREA PIPELINE-ULUI
# ============================================================

print("\nIncepe ingestia datelor MIT-BIH...\n")

X_train, y_train, rr_pre_tr, rr_post_tr = load_split(
    DATA_DIR, DS1_TRAIN, "TRAIN"
)

X_test, y_test, rr_pre_te, rr_post_te = load_split(
    DATA_DIR, DS2_TEST, "TEST"
)

# ============================================================
# 7. VALIDAREA REZULTATELOR INGESTIEI
# ============================================================

assert X_train.ndim == 2 and X_train.shape[1] == 200
assert X_test.ndim == 2 and X_test.shape[1] == 200

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert len(X_train) == len(rr_pre_tr) == len(rr_post_tr)
assert len(X_test) == len(rr_pre_te) == len(rr_post_te)

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

assert np.isfinite(rr_pre_tr).all()
assert np.isfinite(rr_post_tr).all()
assert np.isfinite(rr_pre_te).all()
assert np.isfinite(rr_post_te).all()

valid_classes = set(AAMI_MAP.values())
assert set(np.unique(y_train)).issubset(valid_classes)
assert set(np.unique(y_test)).issubset(valid_classes)

# ============================================================
# 8. REZUMATUL PIPELINE-ULUI DE INGESTIE
# ============================================================

print("\n" + "=" * 58)
print("REZUMAT PIPELINE DE INGESTIE - CP1 / S5")
print("=" * 58)

print(f"Inregistrari train: {len(DS1_TRAIN)}")
print(f"Batai train:        {len(X_train)}")
print(f" forma X_train:     {X_train.shape}")
print(f"Clase train:        {dict(Counter(map(str, y_train)))}")

print()

print(f"Inregistrari test:  {len(DS2_TEST)}")
print(f"Batai test:         {len(X_test)}")
print(f"Forma X_test:       {X_test.shape}")
print(f"Clase test:         {dict(Counter(map(str, y_test)))}")

print("=" * 58)
print("Validarea ingestiei s-a incheiat cu succes.")
print("Datele sunt pregatite pentru etapa urmatoare.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Director date: /content/mit_bih_extracted/mit-bih-arrhythmia-database-1.0.0
Verificarea fisierelor a reusit.
Train: 23 inregistrari
Test:  21 inregistrari

Incepe ingestia datelor MIT-BIH...

TRAIN: 1/23 - 101
TRAIN: 2/23 - 106
TRAIN: 3/23 - 108
TRAIN: 4/23 - 109
TRAIN: 5/23 - 112
TRAIN: 6/23 - 114
TRAIN: 7/23 - 115
TRAIN: 8/23 - 116
TRAIN: 9/23 - 118
TRAIN: 10/23 - 119
TRAIN: 11/23 - 122
TRAIN: 12/23 - 124
TRAIN: 13/23 - 201
TRAIN: 14/23 - 202
TRAIN: 15/23 - 203
TRAIN: 16/23 - 205
TRAIN: 17/23 - 207
TRAIN: 18/23 - 208
TRAIN: 19/23 - 209
TRAIN: 20/23 - 215
TRAIN: 21/23 - 220
TRAIN: 22/23 - 223
TRAIN: 23/23 - 230
TEST: 1/21 - 100
TEST: 2/21 - 103
TEST: 3/21 - 105
TEST: 4/21 - 111
TEST: 5/21 - 113
TEST: 6/21 - 117
TEST: 7/21 - 121
TEST: 8/21 - 123
TEST: 9/21 - 200
TEST: 10/21 - 210
TEST: 11/21 - 212
TEST: 12/21 - 213
TEST: 13/21 - 214
TEST: 14/21 - 219
TEST: 15